# Step 4A: Fine-Tuning Bi-Encoder & Reranker

This notebook implements the fine-tuning pipeline for MediQuery:

1. **Phase 1** — Prepare training data (24K query–golden-chunk pairs + hard negatives)
2. **Phase 2** — Fine-tune the bi-encoder (`BAAI/bge-base-en-v1.5`)
3. **Phase 3** — Rebuild the FAISS index with fine-tuned embeddings
4. **Phase 4** — Fine-tune the cross-encoder reranker (`BAAI/bge-reranker-v2-m3`)
5. **Phase 5** — Retrieval evaluation (pre-trained vs. fine-tuned)

### Prerequisites

This notebook assumes that the **Step 2–3 notebook** has already been run, producing:

| Artifact | Path |
|---|---|
| Chunk corpus | `{DATA_DIR}/all_chunks.json` |
| Pre-trained FAISS index | `{DATA_DIR}/faiss_index/medicare.index` |
| Chunk metadata | `{DATA_DIR}/faiss_index/chunk_metadata.json` |
| Document IDs | `{DATA_DIR}/faiss_index/docids.txt` |
| Embeddings | `{DATA_DIR}/faiss_index/embeddings.npy` |

### Outputs

After running, the following are saved for **Step 4B** (End-to-End RAG Evaluation):

| Artifact | Path |
|---|---|
| Fine-tuned bi-encoder | `{DATA_DIR}/bge-base-mediquery-finetuned/` |
| Fine-tuned FAISS index | `{DATA_DIR}/faiss_index/medicare_finetuned.index` |
| Fine-tuned embeddings | `{DATA_DIR}/faiss_index/embeddings_finetuned.npy` |
| Fine-tuned reranker | `{DATA_DIR}/bge-reranker-mediquery-finetuned/` |

## Install Dependencies

In [62]:
!pip install -q sentence-transformers faiss-cpu pandas scikit-learn

## Mount Google Drive & Path Configuration

All paths match the conventions from the Step 2–3 notebook.

In [63]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

# ─── PATH CONFIGURATION (matches Step 2-3 notebook) ──────────────────────────
DATA_DIR    = '/content/drive/MyDrive/GenAI_project/Data'
CHUNKS_FILE = f'{DATA_DIR}/all_chunks.json'
OUTPUT_DIR  = f'{DATA_DIR}/faiss_index'
INDEX_FILE  = f'{OUTPUT_DIR}/medicare.index'
DOCID_FILE  = f'{OUTPUT_DIR}/docids.txt'
EMBED_FILE  = f'{OUTPUT_DIR}/embeddings.npy'
META_FILE   = f'{OUTPUT_DIR}/chunk_metadata.json'

# ─── TRAINING & EVAL DATA ────────────────────────────────────────────────────
DATASETS_DIR     = f'{DATA_DIR}'
TRAIN_CSV        = f'{DATASETS_DIR}/query_golden_chunk_pairs_full_with_manuals.csv'
# TRAIN_CSV        = f'{DATASETS_DIR}/query_golden_chunk_pairs_full.csv'
EVAL_CSV         = f'{DATASETS_DIR}/rag_final_eval_500_qa_pairs_v2_claude.csv'

# ─── FINE-TUNED MODEL OUTPUT PATHS ───────────────────────────────────────────
FT_BIENCODER_DIR = f'{DATA_DIR}/bge-base-mediquery-finetuned'
FT_RERANKER_DIR  = f'{DATA_DIR}/bge-reranker-mediquery-finetuned'
FT_INDEX_FILE    = f'{OUTPUT_DIR}/medicare_finetuned.index'
FT_EMBED_FILE    = f'{OUTPUT_DIR}/embeddings_finetuned.npy'

os.makedirs(DATASETS_DIR, exist_ok=True)

print(f"Data directory:      {DATA_DIR}")
print(f"FAISS index dir:     {OUTPUT_DIR}")
print(f"Training CSV:        {TRAIN_CSV}")
print(f"Eval CSV:            {EVAL_CSV}")
print(f"FT bi-encoder dir:   {FT_BIENCODER_DIR}")
print(f"FT reranker dir:     {FT_RERANKER_DIR}")

Data directory:      /content/drive/MyDrive/GenAI_project/Data
FAISS index dir:     /content/drive/MyDrive/GenAI_project/Data/faiss_index
Training CSV:        /content/drive/MyDrive/GenAI_project/Data/query_golden_chunk_pairs_full.csv
Eval CSV:            /content/drive/MyDrive/GenAI_project/Data/rag_final_eval_500_qa_pairs_v2_claude.csv
FT bi-encoder dir:   /content/drive/MyDrive/GenAI_project/Data/bge-base-mediquery-finetuned
FT reranker dir:     /content/drive/MyDrive/GenAI_project/Data/bge-reranker-mediquery-finetuned


## Upload Datasets

Upload the two CSV files to `{DATA_DIR}/datasets/` on Google Drive before running the next cell. Alternatively, upload them here with `files.upload()`.

In [65]:
# Option A: If files are already on Google Drive, just verify they exist
for path in [TRAIN_CSV, EVAL_CSV]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  Found: {os.path.basename(path)} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {path}")
        print(f"  → Upload to Google Drive at: {DATASETS_DIR}/")

# Option B: Upload from local machine (uncomment if needed)
# from google.colab import files
# uploaded = files.upload()
# for name, data in uploaded.items():
#     dest = f'{DATASETS_DIR}/{name}'
#     with open(dest, 'wb') as f:
#         f.write(data)
#     print(f"Saved to {dest}")

  Found: query_golden_chunk_pairs_full.csv (22.2 MB)
  Found: rag_final_eval_500_qa_pairs_v2_claude.csv (0.6 MB)


## Load Pre-Trained Artifacts from Step 2–3

In [66]:
import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from collections import Counter

# ─── Load chunk corpus ────────────────────────────────────────────────────────
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    all_chunks = json.load(f)
print(f"Chunks loaded: {len(all_chunks):,}")

# ─── Load chunk metadata ─────────────────────────────────────────────────────
with open(META_FILE, 'r', encoding='utf-8') as f:
    chunk_metadata = json.load(f)
print(f"Metadata loaded: {len(chunk_metadata):,}")

# ─── Load pre-trained FAISS index ─────────────────────────────────────────────
pretrained_index = faiss.read_index(INDEX_FILE)
print(f"Pre-trained FAISS index: {pretrained_index.ntotal:,} vectors, dim={pretrained_index.d}")

# ─── Device setup ─────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nDevice: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Chunks loaded: 8,563
Metadata loaded: 8,563
Pre-trained FAISS index: 8,563 vectors, dim=768

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB


## Helper Functions

Retrieval and reranking utilities used throughout the notebook. These mirror the functions from the Step 2–3 notebook but accept model/index as parameters so we can swap between pre-trained and fine-tuned versions.

In [ ]:
def retrieve_chunks(query, embed_model, index, all_chunks, chunk_metadata, top_k=100):
    """Embed a query and retrieve top-k chunks from a FAISS index."""
    query_vec = embed_model.encode(
        query, normalize_embeddings=True, convert_to_numpy=True
    ).astype('float32').reshape(1, -1)

    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        results.append({
            'faiss_score': float(scores[0][rank]),
            'text':        all_chunks[idx]['text'],
            'title':       meta['title'],
            'type':        meta['type'],
            'states':      meta.get('states', ['ALL']),
            'source_id':   meta['source_id'],
            'chunk_idx':   meta['chunk_idx'],
            'chunk_id':    f"{meta['source_id']}_{meta['chunk_idx']}"
        })
    return results


def filter_by_state(results, state=None):
    """Filter retrieved chunks to those covering a specific state."""
    if state is None:
        return results
    return [r for r in results if 'ALL' in r['states'] or state in r['states']]


def filter_claims_processing(results, query):
    coverage_keywords = ['cover', 'covered', 'coverage', 'eligible',
                         'eligibility', 'qualify', 'benefit']
    is_coverage_query = any(kw in query.lower() for kw in coverage_keywords)
    if is_coverage_query:
        return [r for r in results if r['type'] != 'CLAIMS_PROCESSING']
    return results  # keep everything for billing/documentation queries


def rerank_results(query, results, reranker_model, top_n=5, deduplicate=True):
    """Rerank retrieved chunks using a cross-encoder.

    Args:
        deduplicate: If True, keep only one chunk per source_id (for RAG generation).
                     If False, keep all chunks (for retrieval evaluation).
    """
    if not results:
        return []

    pairs = [(query, r['text']) for r in results]
    scores = reranker_model.predict(pairs)

    for i in range(len(results)):
        results[i]['rerank_score'] = float(scores[i])

    results.sort(key=lambda x: x['rerank_score'], reverse=True)

    if not deduplicate:
        return results[:top_n]

    seen = set()
    final = []
    for r in results:
        if r['source_id'] not in seen:
            final.append(r)
            seen.add(r['source_id'])
        if len(final) == top_n:
            break
    return final


def retrieve_and_rerank(query, embed_model, faiss_index, reranker_model,
                        all_chunks, chunk_metadata, state=None,
                        top_k_retrieve=100, top_n_rerank=5):
    """Full retrieval pipeline: embed -> FAISS -> state filter -> filter claims -> rerank."""
    retrieved = retrieve_chunks(query, embed_model, faiss_index,
                                all_chunks, chunk_metadata, top_k=top_k_retrieve)
    filtered = filter_by_state(retrieved, state)
    filtered = filter_claims_processing(filtered, query)
    reranked = rerank_results(query, filtered, reranker_model, top_n=top_n_rerank)
    return reranked


print('Helper functions defined.')

Helper functions defined.


---
# Phase 1: Prepare Training Data

Load the 24K query–golden-chunk pairs, construct hard negatives using the pre-trained FAISS index, and split into train/validation sets.

### Step 1.1 — Load and inspect the 24K pairs

In [68]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
print(f"Total training pairs: {len(train_df):,}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\n--- Query type distribution ---")
print(train_df['query_type'].value_counts())
print(f"\n--- Document type distribution ---")
print(train_df['type'].value_counts())
print(f"\n--- Sample row ---")
train_df.head(1)

Total training pairs: 24,032

Columns: ['query_id', 'query', 'query_type', 'gold_chunk_id', 'source_id', 'chunk_idx', 'title', 'type', 'coverage_status', 'states', 'filename', 'gold_chunk_excerpt', 'reference_answer']

--- Query type distribution ---
query_type
policy                 8645
yes_no                 3096
summary                3096
ncd_reference          3096
use_case               2386
service_condition      1384
indications             976
criteria                928
mixed_policy            302
non_coverage_reason      67
retired_status           56
Name: count, dtype: int64

--- Document type distribution ---
type
LCD    19365
NCD     4667
Name: count, dtype: int64

--- Sample row ---


,query_id,query,query_type,gold_chunk_id,source_id,chunk_idx,title,type,coverage_status,states,filename,gold_chunk_excerpt,reference_answer
0,100.3_0_q1,What is the Medicare coverage policy for 24-Ho...,policy,100.3_0,100.3,0,24-Hour Ambulatory Esophageal pH Monitoring,NCD,covered,ALL,24-Hour_Ambulatory_Esophageal_pH_Monitoring.txt,Twenty-four hour ambulatory pH monitoring is c...,Twenty-four hour ambulatory pH monitoring is c...


### Step 1.2 — Build (query, positive) pairs using actual indexed chunk text

**Critical:** The positive passage must be the **actual chunk text from `all_chunks.json`** (the same text that is embedded in the FAISS index), NOT the `gold_chunk_excerpt` from the CSV. The indexed chunks include metadata headers (e.g., `[TYPE: NCD | NCD_ID: 30.3.3 | ...]`) and are ~400 words long. Training on short excerpts would create a domain mismatch — the model would learn to embed short text well but fail on the long metadata-tagged chunks in the actual index.

In [69]:
# Build a lookup: chunk_id -> index in all_chunks for fast negative mining
chunk_id_to_idx = {}
for i, meta in enumerate(chunk_metadata):
    cid = f"{meta['source_id']}_{meta['chunk_idx']}"
    chunk_id_to_idx[cid] = i

print(f"Chunk ID lookup built: {len(chunk_id_to_idx):,} entries")

# Build training pairs using ACTUAL INDEXED CHUNK TEXT (not the short CSV excerpt)
train_pairs = []
skipped = 0
skipped_no_chunk = 0

for _, row in train_df.iterrows():
    gold_chunk_id = str(row['gold_chunk_id'])

    # Look up the actual chunk text from all_chunks.json
    if gold_chunk_id in chunk_id_to_idx:
        chunk_idx = chunk_id_to_idx[gold_chunk_id]
        positive_text = all_chunks[chunk_idx]['text']  # full metadata-tagged chunk
    else:
        # Chunk not found in index — skip this pair
        skipped_no_chunk += 1
        continue

    if len(positive_text.strip()) < 50:
        skipped += 1
        continue

    train_pairs.append({
        'query':          row['query'],
        'positive':       positive_text,
        'gold_chunk_id':  gold_chunk_id,
        'source_id':      str(row['source_id']),
        'query_type':     row['query_type'],
    })

print(f"Valid training pairs: {len(train_pairs):,}")
print(f"Skipped (chunk not in index): {skipped_no_chunk}")
print(f"Skipped (short text): {skipped}")
print(f"\nSample positive text (first 200 chars):")
print(f"  {train_pairs[0]['positive'][:200]}...")

Chunk ID lookup built: 8,563 entries
Valid training pairs: 24,032
Skipped (chunk not in index): 0
Skipped (short text): 0

Sample positive text (first 200 chars):
  [TYPE: NCD | NCD_ID: 100.3 | TITLE: 24-Hour Ambulatory Esophageal pH Monitoring]

TITLE: 24-Hour Ambulatory Esophageal pH Monitoring SECTION: 100.3 EFFECTIVE DATE: 1985-06-11 00:00:00 COVERAGE POLICY:...


### Step 1.3 — Mine hard negatives from the pre-trained FAISS index

For each query, retrieve the top-20 chunks using the **pre-trained** FAISS index. The highest-ranked chunk whose `source_id` differs from the gold `source_id` becomes the hard negative. This teaches the model to distinguish between semantically similar but irrelevant passages.

In [70]:
from tqdm import tqdm

# Load the pre-trained embedding model for negative mining
pretrained_embed_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

print(f"Mining hard negatives for {len(train_pairs):,} queries...")
print("This encodes each query and searches the pre-trained FAISS index.\n")

# Batch-encode all queries for efficiency
all_queries = [p['query'] for p in train_pairs]
query_embeddings = pretrained_embed_model.encode(
    all_queries,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print(f"Query embeddings shape: {query_embeddings.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mining hard negatives for 24,032 queries...
This encodes each query and searches the pre-trained FAISS index.



Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Query embeddings shape: (24032, 768)


In [ ]:
# Batch FAISS search for all queries at once
TOP_K_MINE = 20
scores_all, indices_all = pretrained_index.search(query_embeddings, TOP_K_MINE)

# Assign hard negatives
no_negative_count = 0
for i, pair in enumerate(tqdm(train_pairs, desc="Assigning hard negatives")):
    gold_source = pair['source_id']
    negative_text = None

    for idx in indices_all[i]:
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        if str(meta['source_id']) != gold_source:
            negative_text = all_chunks[idx]['text']
            break

    if negative_text is None:
        # Fallback: use the last retrieved chunk
        fallback_idx = indices_all[i][-1]
        if fallback_idx != -1:
            negative_text = all_chunks[fallback_idx]['text']
        else:
            negative_text = ""  # will be filtered out
        no_negative_count += 1

    pair['negative'] = negative_text

# Filter out any pairs with empty negatives
train_pairs = [p for p in train_pairs if len(p.get('negative', '')) > 20]

print(f"\nFinal training pairs with negatives: {len(train_pairs):,}")
print(f"Pairs where no distinct-source negative was found: {no_negative_count}")

Assigning hard negatives: 100%|██████████| 24032/24032 [00:00<00:00, 1096030.20it/s]


Final training pairs with negatives: 24,032
Pairs where no distinct-source negative was found: 0


### Step 1.4 — Train / validation split

In [72]:
from sklearn.model_selection import train_test_split

# Stratify by query_type to keep distribution balanced
query_types = [p['query_type'] for p in train_pairs]

train_set, val_set = train_test_split(
    train_pairs,
    test_size=0.05,
    random_state=42,
    stratify=query_types
)

print(f"Train set: {len(train_set):,}")
print(f"Val set:   {len(val_set):,}")

Train set: 22,830
Val set:   1,202


### Step 1.5 — Add manual chapter pairs to training data

The 24K training pairs come exclusively from NCDs and LCDs. The fine-tuned model becomes blind to **Claims Processing Manual** and **Benefit Policy Manual** chunks because it never sees them during training. We pull manual chapter queries from the 500-item eval CSV and add them as extra training pairs so the model learns what those chunks look like too.

In [ ]:
# # Load the eval CSV to get manual chapter questions
# eval_df_early = pd.read_csv(EVAL_CSV)

# manual_extra_pairs = []

# for _, row in eval_df_early.iterrows():
#     if row['doc_type'] not in ['Medicare Claims Processing Manual',
#                                 'Medicare Benefit Policy Manual']:
#         continue

#     chunk_id = str(row['chunk_id'])
#     if chunk_id not in chunk_id_to_idx:
#         continue

#     idx = chunk_id_to_idx[chunk_id]
#     positive_text = all_chunks[idx]['text']

#     # Mine a hard negative using the pre-trained FAISS index
#     query_vec = pretrained_embed_model.encode(
#         row['question'], normalize_embeddings=True, convert_to_numpy=True
#     ).astype('float32').reshape(1, -1)

#     _, neg_indices = pretrained_index.search(query_vec, 20)
#     negative_text = None
#     gold_source = chunk_id.rsplit('_', 1)[0]

#     for neg_idx in neg_indices[0]:
#         if neg_idx == -1:
#             continue
#         if str(chunk_metadata[neg_idx]['source_id']) != gold_source:
#             negative_text = all_chunks[neg_idx]['text']
#             break

#     if negative_text is None:
#         continue

#     manual_extra_pairs.append({
#         'query':         row['question'],
#         'positive':      positive_text,
#         'negative':      negative_text,
#         'source_id':     gold_source,
#         'query_type':    row['question_type'],
#         'gold_chunk_id': chunk_id,
#     })

# print(f"Extra manual chapter pairs added: {len(manual_extra_pairs)}")

# # Combine with original train_set
# combined_train_set = train_set + manual_extra_pairs
# print(f"Combined training set: {len(combined_train_set):,}")

Extra manual chapter pairs added: 126
Combined training set: 22,956


### Step 1.6 — Create InputExample objects for bi-encoder training

In [ ]:
from sentence_transformers import InputExample

# Triplets: (query, positive, hard_negative)
biencoder_train_examples = [
    InputExample(texts=[p['query'], p['positive'], p['negative']])
    for p in train_set
]

print(f"Bi-encoder training examples: {len(biencoder_train_examples):,}")
print(f"  Sample query:    {biencoder_train_examples[0].texts[0][:120]}...")
print(f"  Sample positive: {biencoder_train_examples[0].texts[1][:120]}...")
print(f"  Sample negative: {biencoder_train_examples[0].texts[2][:120]}...")

Bi-encoder training examples: 22,956
  Sample query:    What does NCD 220.6.2 say about FDG PET for Lung Cancer (Replaced with Section 220.6.17)?...
  Sample positive: [TYPE: NCD | NCD_ID: 220.6.2 | TITLE: FDG PET for Lung Cancer (Replaced with Section 220.6.17) - RETIRED]

TITLE: FDG PE...
  Sample negative: [TYPE: NCD | NCD_ID: 220.6.10 | TITLE: FDG PET for Breast Cancer (Replaced with Section 220.6.17) - RETIRED]

TITLE: FDG...


---
# Phase 2: Fine-Tune the Bi-Encoder

Fine-tune `BAAI/bge-base-en-v1.5` using **MultipleNegativesRankingLoss (MNRL)** — the standard contrastive loss for bi-encoder training. MNRL treats other in-batch positives as additional negatives, maximizing training signal.

### Step 2.1 — Load base model and configure loss

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

# Load fresh base model for fine-tuning
biencoder_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
print(f"Base model loaded: BAAI/bge-base-en-v1.5")
print(f"Embedding dim: {biencoder_model.get_sentence_embedding_dimension()}")

# Configure loss
train_loss = MultipleNegativesRankingLoss(biencoder_model)
print(f"Loss: MultipleNegativesRankingLoss")

# Create DataLoader
BATCH_SIZE = 128  # Adjust: T4 → 32, A100 → 64-128
train_dataloader = DataLoader(
    biencoder_train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE
)
print(f"DataLoader: {len(train_dataloader)} batches of {BATCH_SIZE}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base model loaded: BAAI/bge-base-en-v1.5
Embedding dim: 768
Loss: MultipleNegativesRankingLoss
DataLoader: 718 batches of 32


### Step 2.2 — Set up IR evaluator against the full corpus

The evaluator must search the full chunk corpus (all 8,563 chunks), not just the validation positives. Using only val positives as the corpus makes evaluation trivially easy — the model only has to pick the right answer from ~1,200 candidates instead of 8,563.

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Build the full corpus from all_chunks — every chunk the FAISS index contains.
full_ir_corpus = {str(i): chunk['text'] for i, chunk in enumerate(all_chunks)}

# Queries and relevant docs stay the same — we still use val_set queries,
# but now each query has to find its gold chunk among all 8563, not just ~1202.
ir_queries       = {str(i): p['query'] for i, p in enumerate(val_set)}
ir_relevant_docs = {str(i): {str(chunk_id_to_idx[p['gold_chunk_id']])}
                    for i, p in enumerate(val_set)}

biencoder_evaluator = InformationRetrievalEvaluator(
    ir_queries,
    full_ir_corpus,
    ir_relevant_docs,
    name='mediquery-biencoder-val',
    show_progress_bar=True
)

print(f"IR Evaluator: {len(ir_queries)} queries against {len(full_ir_corpus)} corpus docs")

IR Evaluator: 1202 queries against 8563 corpus docs


### Step 2.3 — Train the bi-encoder

| Parameter | Value | Rationale |
|---|---|---|
| Epochs | 2 | Conservative to avoid overfitting on 24K pairs |
| Learning rate | 1e-5 | Lower LR prevents catastrophic forgetting of pre-trained representations |
| Batch size | 32 | Larger = more in-batch negatives for MNRL |
| Warmup steps | 200 | ~1% of total steps |

In [ ]:
EPOCHS = 2
WARMUP_STEPS = 200
EVAL_STEPS = 500
LR = 1e-5

total_steps = len(train_dataloader) * EPOCHS
print(f"Training plan:")
print(f"  Epochs:       {EPOCHS}")
print(f"  Steps/epoch:  {len(train_dataloader)}")
print(f"  Total steps:  {total_steps}")
print(f"  Warmup:       {WARMUP_STEPS}")
print(f"  LR:           {LR}")
print(f"  Eval every:   {EVAL_STEPS} steps")
print(f"  Output:       {FT_BIENCODER_DIR}")
print(f"\nStarting training...\n")

biencoder_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=biencoder_evaluator,
    epochs=EPOCHS,
    evaluation_steps=EVAL_STEPS,
    warmup_steps=WARMUP_STEPS,
    output_path=FT_BIENCODER_DIR,
    save_best_model=True,
    optimizer_params={'lr': LR},
    use_amp=True
)

print(f"\nBi-encoder fine-tuning complete!")
print(f"Best model saved to: {FT_BIENCODER_DIR}")

Training plan:
  Epochs:       3
  Steps/epoch:  718
  Total steps:  2154
  Warmup:       200
  LR:           1e-05
  Eval every:   500 steps
  Output:       /content/drive/MyDrive/GenAI_project/Data/bge-base-mediquery-finetuned

Starting training...



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Mediquery-biencoder-val Cosine Accuracy@1,Mediquery-biencoder-val Cosine Accuracy@3,Mediquery-biencoder-val Cosine Accuracy@5,Mediquery-biencoder-val Cosine Accuracy@10,Mediquery-biencoder-val Cosine Precision@1,Mediquery-biencoder-val Cosine Precision@3,Mediquery-biencoder-val Cosine Precision@5,Mediquery-biencoder-val Cosine Precision@10,Mediquery-biencoder-val Cosine Recall@1,Mediquery-biencoder-val Cosine Recall@3,Mediquery-biencoder-val Cosine Recall@5,Mediquery-biencoder-val Cosine Recall@10,Mediquery-biencoder-val Cosine Ndcg@10,Mediquery-biencoder-val Cosine Mrr@10,Mediquery-biencoder-val Cosine Map@100
500,0.485038,No log,0.271215,0.532446,0.677205,0.833611,0.271215,0.177482,0.135441,0.083361,0.271215,0.532446,0.677205,0.833611,0.530811,0.435975,0.445547
718,0.485038,No log,0.261231,0.524126,0.680532,0.841098,0.261231,0.174709,0.136106,0.084110,0.261231,0.524126,0.680532,0.841098,0.529174,0.431432,0.440440
1000,0.295996,No log,0.270383,0.540765,0.692180,0.840266,0.270383,0.180255,0.138436,0.084027,0.270383,0.540765,0.692180,0.840266,0.535935,0.440333,0.449382
1436,0.295996,No log,0.272047,0.540765,0.696339,0.840266,0.272047,0.180255,0.139268,0.084027,0.272047,0.540765,0.696339,0.840266,0.539105,0.444314,0.453412
1500,0.262145,No log,0.276206,0.534110,0.688852,0.842762,0.276206,0.178037,0.137770,0.084276,0.276206,0.534110,0.688852,0.842762,0.539435,0.444297,0.453364
2000,0.244346,No log,0.277038,0.556572,0.700499,0.845258,0.277038,0.185524,0.140100,0.084526,0.277038,0.556572,0.700499,0.845258,0.544719,0.450080,0.459034
2154,0.244346,No log,0.274542,0.561564,0.698835,0.841930,0.274542,0.187188,0.139767,0.084193,0.274542,0.561564,0.698835,0.841930,0.542849,0.448520,0.457752


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/268 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:05<00:00,  5.01s/it]



Bi-encoder fine-tuning complete!
Best model saved to: /content/drive/MyDrive/GenAI_project/Data/bge-base-mediquery-finetuned


### Step 2.4 — Load and verify the fine-tuned bi-encoder

In [78]:
finetuned_embed_model = SentenceTransformer(FT_BIENCODER_DIR, device=device)
print(f"Fine-tuned bi-encoder loaded from: {FT_BIENCODER_DIR}")
print(f"Embedding dim: {finetuned_embed_model.get_sentence_embedding_dimension()}")

# Quick verification: encode a test query
test_emb = finetuned_embed_model.encode("test", normalize_embeddings=True)
print(f"L2 norm check: {np.linalg.norm(test_emb):.6f} (should be 1.0)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned bi-encoder loaded from: /content/drive/MyDrive/GenAI_project/Data/bge-base-mediquery-finetuned
Embedding dim: 768
L2 norm check: 1.000000 (should be 1.0)


---
# Phase 3: Rebuild FAISS Index with Fine-Tuned Embeddings

### Step 3.1 — Re-encode all 8,563 chunks

In [ ]:
texts = [chunk['text'] for chunk in all_chunks]

print(f"Re-encoding {len(texts):,} chunks with fine-tuned bi-encoder...")

finetuned_embeddings = finetuned_embed_model.encode(
    texts,
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {finetuned_embeddings.shape}")
print(f"Dtype: {finetuned_embeddings.dtype}")

norms = np.linalg.norm(finetuned_embeddings, axis=1)
print(f"L2 norm check — mean: {norms.mean():.6f}, min: {norms.min():.6f}, max: {norms.max():.6f}")

Re-encoding 8,563 chunks with fine-tuned bi-encoder...


Batches:   0%|          | 0/134 [00:00<?, ?it/s]


Embeddings shape: (8563, 768)
Dtype: float32
L2 norm check — mean: 1.000000, min: 1.000000, max: 1.000000


### Step 3.2 — Build and save the fine-tuned FAISS index

In [80]:
dimension = finetuned_embeddings.shape[1]
finetuned_index = faiss.IndexFlatIP(dimension)
finetuned_index.add(finetuned_embeddings.astype('float32'))

faiss.write_index(finetuned_index, FT_INDEX_FILE)
np.save(FT_EMBED_FILE, finetuned_embeddings)

print(f"Fine-tuned FAISS index saved: {FT_INDEX_FILE}")
print(f"  Vectors: {finetuned_index.ntotal:,}")
print(f"  Dimension: {finetuned_index.d}")
print(f"Fine-tuned embeddings saved: {FT_EMBED_FILE}")
print(f"  Size: {os.path.getsize(FT_INDEX_FILE) / 1024**2:.1f} MB")

Fine-tuned FAISS index saved: /content/drive/MyDrive/GenAI_project/Data/faiss_index/medicare_finetuned.index
  Vectors: 8,563
  Dimension: 768
Fine-tuned embeddings saved: /content/drive/MyDrive/GenAI_project/Data/faiss_index/embeddings_finetuned.npy
  Size: 25.1 MB


### Step 3.3 — Sanity check: compare retrieval on 3 test queries

In [81]:
test_queries = [
    "Does Medicare cover acupuncture for chronic lower back pain?",
    "Is home oxygen therapy covered for patients in Texas?",
    "What are the requirements for skilled nursing facility coverage?"
]

print("=" * 95)
print("RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned Bi-Encoder (top-3)")
print("=" * 95)

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 95)

    # Pre-trained
    pt_results = retrieve_chunks(query, pretrained_embed_model, pretrained_index,
                                  all_chunks, chunk_metadata, top_k=3)
    # Fine-tuned
    ft_results = retrieve_chunks(query, finetuned_embed_model, finetuned_index,
                                  all_chunks, chunk_metadata, top_k=3)

    print(f"  {'Pre-trained':^45s} | {'Fine-tuned':^45s}")
    print(f"  {'-'*45} | {'-'*45}")
    for rank in range(3):
        pt = pt_results[rank] if rank < len(pt_results) else {'title': '—', 'faiss_score': 0}
        ft = ft_results[rank] if rank < len(ft_results) else {'title': '—', 'faiss_score': 0}
        pt_str = f"#{rank+1} {pt['title'][:30]:30s} ({pt['faiss_score']:.4f})"
        ft_str = f"#{rank+1} {ft['title'][:30]:30s} ({ft['faiss_score']:.4f})"
        print(f"  {pt_str:45s} | {ft_str:45s}")

RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned Bi-Encoder (top-3)

Query: "Does Medicare cover acupuncture for chronic lower back pain?"
-----------------------------------------------------------------------------------------------
                   Pre-trained                  |                  Fine-tuned                  
  --------------------------------------------- | ---------------------------------------------
  #1 Acupuncture for Chronic Lower  (0.8095)    | #1 Acupuncture for Chronic Lower  (0.7217)   
  #2 Acupuncture                    (0.7742)    | #2 Acupuncture                    (0.5107)   
  #3 Acupuncture for Fibromyalgia   (0.7358)    | #3 Spinal Cord Stimulators for Ch (0.4982)   

Query: "Is home oxygen therapy covered for patients in Texas?"
-----------------------------------------------------------------------------------------------
                   Pre-trained                  |                  Fine-tuned                  
  ---------------------------

---
# Phase 4: Fine-Tune the Cross-Encoder Reranker

The reranker is the final precision gate — it decides which 5 chunks reach the LLM. Fine-tuning it on Medicare-specific relevance judgments teaches the model to better distinguish between truly relevant policy text and semantically similar but irrelevant passages.

### Step 4.1 — Re-mine hard negatives from the fine-tuned FAISS index

The bi-encoder hard negatives mined in Phase 1 came from the **pre-trained** index. Now that we have a fine-tuned index, we re-mine harder negatives — passages the fine-tuned bi-encoder ranks highly but that come from a different source than the gold chunk.

We mine **10 hard negatives per query** (up from 1). Cross-encoders score `(query, passage)` pairs independently — they don’t benefit from in-batch negatives like bi-encoders do — so providing more explicit negatives gives a much stronger training signal. Each query produces 1 positive + 10 negative examples = 11 training examples.

In [ ]:
# Re-mine hard negatives using the fine-tuned bi-encoder + fine-tuned FAISS index
# Mine 10 hard negatives per query for stronger reranker training signal.
# Use combined_train_set so reranker sees manual (Claims/Benefit) pairs too.
reranker_train_set = combined_train_set if 'combined_train_set' in globals() else train_set

print("Re-mining hard negatives from the fine-tuned FAISS index...\n")

# Batch-encode all training queries with the fine-tuned model
train_queries = [p['query'] for p in reranker_train_set]
ft_query_embeddings = finetuned_embed_model.encode(
    train_queries,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

# Search fine-tuned FAISS index — retrieve enough candidates to find 10 from different sources
NUM_HARD_NEGATIVES = 10
TOP_K_MINE_FT = 50  # retrieve 50 to ensure we can find 10 distinct-source negatives
scores_ft, indices_ft = finetuned_index.search(ft_query_embeddings, TOP_K_MINE_FT)

# Assign up to 10 hard negatives per training query
neg_counts = []
for i, pair in enumerate(tqdm(reranker_train_set, desc="Mining FT hard negatives")):
    gold_source = pair['source_id']
    negatives = []
    seen_sources = {gold_source}  # skip gold source and avoid duplicates

    for idx in indices_ft[i]:
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        src = str(meta['source_id'])
        if src in seen_sources:
            continue
        seen_sources.add(src)
        negatives.append(all_chunks[idx]['text'])
        if len(negatives) == NUM_HARD_NEGATIVES:
            break

    # Fallback: if fewer than 10 found, pad with non-source-deduped candidates
    if len(negatives) < NUM_HARD_NEGATIVES:
        for idx in indices_ft[i]:
            if idx == -1:
                continue
            meta = chunk_metadata[idx]
            if str(meta['source_id']) == gold_source:
                continue
            text = all_chunks[idx]['text']
            if text not in negatives:
                negatives.append(text)
            if len(negatives) == NUM_HARD_NEGATIVES:
                break

    pair['ft_negatives'] = negatives  # list of up to 10 negatives
    pair['ft_negative'] = negatives[0] if negatives else pair['negative']  # backward compat
    neg_counts.append(len(negatives))

print(f"\nRe-mined hard negatives for {len(reranker_train_set):,} training pairs")
print(f"Negatives per query: min={min(neg_counts)}, max={max(neg_counts)}, avg={sum(neg_counts)/len(neg_counts):.1f}")

# Also re-mine for validation set
val_queries = [p['query'] for p in val_set]
ft_val_embeddings = finetuned_embed_model.encode(
    val_queries,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

scores_val_ft, indices_val_ft = finetuned_index.search(ft_val_embeddings, TOP_K_MINE_FT)

for i, pair in enumerate(tqdm(val_set, desc="Mining FT val hard negatives")):
    gold_source = pair['source_id']
    negatives = []
    seen_sources = {gold_source}

    for idx in indices_val_ft[i]:
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        src = str(meta['source_id'])
        if src in seen_sources:
            continue
        seen_sources.add(src)
        negatives.append(all_chunks[idx]['text'])
        if len(negatives) == NUM_HARD_NEGATIVES:
            break

    pair['ft_negatives'] = negatives
    pair['ft_negative'] = negatives[0] if negatives else pair['negative']

# Build reranker training examples: 1 positive + 10 negatives per query
reranker_train_examples = []
for pair in reranker_train_set:
    reranker_train_examples.append(
        InputExample(texts=[pair['query'], pair['positive']], label=1.0)
    )
    for neg_text in pair['ft_negatives']:
        reranker_train_examples.append(
            InputExample(texts=[pair['query'], neg_text], label=0.0)
        )

print(f"\nReranker training examples: {len(reranker_train_examples):,}")
print(f"  ({len(reranker_train_set):,} queries x ~{NUM_HARD_NEGATIVES+1} examples each)")

# Validation data: 1 positive + 10 negatives per query
reranker_val_examples = []
for pair in val_set:
    reranker_val_examples.append(
        InputExample(texts=[pair['query'], pair['positive']], label=1.0)
    )
    for neg_text in pair['ft_negatives']:
        reranker_val_examples.append(
            InputExample(texts=[pair['query'], neg_text], label=0.0)
        )

print(f"Reranker validation examples: {len(reranker_val_examples):,}")


### Step 4.2 — Load base cross-encoder and create DataLoader

In [ ]:
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator

# Load base reranker
reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
print(f"Base reranker loaded: BAAI/bge-reranker-v2-m3")

# Create DataLoader
RERANKER_BATCH_SIZE = 32  # Cross-encoders use more memory; 16 is safe on T4
reranker_dataloader = DataLoader(
    reranker_train_examples,
    shuffle=True,
    batch_size=RERANKER_BATCH_SIZE
)
print(f"DataLoader: {len(reranker_dataloader)} batches of {RERANKER_BATCH_SIZE}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Base reranker loaded: BAAI/bge-reranker-v2-m3
DataLoader: 2854 batches of 16


### Step 4.3 — Set up reranker evaluation callback

In [ ]:
# Build eval samples for CERerankingEvaluator (using fine-tuned negatives)
reranker_eval_samples = []
for pair in val_set:
    reranker_eval_samples.append({
        'query':    pair['query'],
        'positive': [pair['positive']],
        'negative': pair['ft_negatives']  # all 10 hard negatives
    })

reranker_evaluator = CERerankingEvaluator(
    reranker_eval_samples,
    name='mediquery-reranker-val'
)

print(f"Reranker evaluator: {len(reranker_eval_samples)} samples, {NUM_HARD_NEGATIVES} negatives each")


### Step 4.4 — Train the reranker

| Parameter | Value | Rationale |
|---|---|---|
| Epochs | 2 | Cross-encoders overfit faster than bi-encoders |
| Learning rate | 5e-6 | Conservative to preserve pre-trained quality |
| Batch size | 16 | Cross-encoders are memory-heavy |
| Warmup steps | 100 | Short warmup is sufficient |

In [85]:
RERANKER_EPOCHS = 2
RERANKER_WARMUP = 100
RERANKER_EVAL_STEPS = 500
RERANKER_LR = 5e-6

total_reranker_steps = len(reranker_dataloader) * RERANKER_EPOCHS
print(f"Reranker training plan:")
print(f"  Epochs:       {RERANKER_EPOCHS}")
print(f"  Steps/epoch:  {len(reranker_dataloader)}")
print(f"  Total steps:  {total_reranker_steps}")
print(f"  Warmup:       {RERANKER_WARMUP}")
print(f"  LR:           {RERANKER_LR}")
print(f"  Output:       {FT_RERANKER_DIR}")
print(f"\nStarting reranker training...\n")

reranker_model.fit(
    train_dataloader=reranker_dataloader,
    evaluator=reranker_evaluator,
    epochs=RERANKER_EPOCHS,
    evaluation_steps=RERANKER_EVAL_STEPS,
    warmup_steps=RERANKER_WARMUP,
    output_path=FT_RERANKER_DIR,
    save_best_model=True,
    optimizer_params={'lr': RERANKER_LR},
    use_amp=True
)

print(f"\nReranker fine-tuning complete!")
print(f"Best model saved to: {FT_RERANKER_DIR}")

Reranker training plan:
  Epochs:       2
  Steps/epoch:  2854
  Total steps:  5708
  Warmup:       100
  LR:           5e-06
  Output:       /content/drive/MyDrive/GenAI_project/Data/bge-reranker-mediquery-finetuned

Starting reranker training...



Token indices sequence length is longer than the specified maximum sequence length for this model (585 > 512). Running this sequence through the model will result in indexing errors


Step,Training Loss,Validation Loss,Mediquery-reranker-val Map,Mediquery-reranker-val Mrr@10,Mediquery-reranker-val Ndcg@10
500,0.434891,No log,0.954243,0.954243,0.966225
1000,0.319583,No log,0.964226,0.964226,0.973594
1500,0.260629,No log,0.968386,0.968386,0.976664
2000,0.239543,No log,0.972962,0.972962,0.980042
2500,0.235292,No log,0.969218,0.969218,0.977279
3000,0.209998,No log,0.973378,0.973378,0.980349
3500,0.196425,No log,0.975042,0.975042,0.981577
4000,0.197367,No log,0.974210,0.974210,0.980963
4500,0.196755,No log,0.972130,0.972130,0.979428
5000,0.176008,No log,0.971298,0.971298,0.978814


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Reranker fine-tuning complete!
Best model saved to: /content/drive/MyDrive/GenAI_project/Data/bge-reranker-mediquery-finetuned


### Step 4.5 — Load and verify the fine-tuned reranker

In [86]:
finetuned_reranker = CrossEncoder(FT_RERANKER_DIR, device=device, max_length=512)
print(f"Fine-tuned reranker loaded from: {FT_RERANKER_DIR}")

# Sanity check: score a relevant vs irrelevant pair
# Use a real chunk from the corpus for the test
acupuncture_chunk = None
wound_chunk = None
for c in all_chunks:
    if '30.3.3' in c.get('text', '') and acupuncture_chunk is None:
        acupuncture_chunk = c['text'][:500]
    if 'Wound Care' in c.get('text', '') and wound_chunk is None:
        wound_chunk = c['text'][:500]
    if acupuncture_chunk and wound_chunk:
        break

test_query = "Does Medicare cover acupuncture for chronic lower back pain?"
scores = finetuned_reranker.predict([
    (test_query, acupuncture_chunk),
    (test_query, wound_chunk)
])

print(f"\nSanity check — query: \"{test_query}\"")
print(f"  Relevant (acupuncture NCD):  {scores[0]:.4f}")
print(f"  Irrelevant (wound care LCD): {scores[1]:.4f}")
print(f"  Relevant > Irrelevant: {scores[0] > scores[1]}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Fine-tuned reranker loaded from: /content/drive/MyDrive/GenAI_project/Data/bge-reranker-mediquery-finetuned

Sanity check — query: "Does Medicare cover acupuncture for chronic lower back pain?"
  Relevant (acupuncture NCD):  0.9645
  Irrelevant (wound care LCD): 0.0000
  Relevant > Irrelevant: True


---
# Phase 5: Retrieval Evaluation (Pre-trained vs. Fine-tuned)

Evaluate retrieval quality on the held-out **500 QA eval set** using Recall@k and MRR.

### Step 5.1 — Load the 500-question eval set

In [87]:
eval_df = pd.read_csv(EVAL_CSV)
print(f"Evaluation set loaded: {len(eval_df)} questions")
print(f"\nColumns: {list(eval_df.columns)}")
print(f"\n--- Question type distribution ---")
print(eval_df['question_type'].value_counts())
print(f"\n--- Document type distribution ---")
print(eval_df['doc_type'].value_counts())
print(f"\n--- Coverage status distribution ---")
print(eval_df['coverage_status'].value_counts())

Evaluation set loaded: 500 questions

Columns: ['qa_id', 'question', 'question_type', 'answer', 'source_title', 'source_type', 'doc_type', 'coverage_status', 'chunk_id', 'source_file', 'states', 'filename']

--- Question type distribution ---
question_type
indications            277
criteria                99
policy                  75
mixed_policy            24
non_coverage_reason     20
retired_status           5
Name: count, dtype: int64

--- Document type distribution ---
doc_type
LCD                                  201
NCD                                  149
Medicare Claims Processing Manual     99
Medicare Benefit Policy Manual        51
Name: count, dtype: int64

--- Coverage status distribution ---
coverage_status
informational    390
covered           61
mixed             24
non-covered       20
retired            5
Name: count, dtype: int64


### Step 5.2 — Evaluate retrieval for both models

In [ ]:
import math

def ndcg_at_k(gold_id, ranked_ids, k):
    """Compute NDCG@k for a single query with one relevant document (binary relevance).

    With a single relevant doc the ideal DCG is always 1.0 (relevant doc at rank 1),
    so NDCG@k = 1/log2(rank+1) if the gold doc appears in the top-k, else 0.
    """
    try:
        rank = ranked_ids[:k].index(gold_id)  # 0-based
        return 1.0 / math.log2(rank + 2)      # +2 because rank is 0-based
    except ValueError:
        return 0.0


def evaluate_retrieval(embed_model, faiss_index, reranker_model,
                       all_chunks, chunk_metadata, eval_df,
                       top_k_retrieve=100, top_n_rerank=5,
                       apply_claims_filter=False):
    """
    Evaluate retrieval performance on the eval set.

    Returns a dict with Recall@k, MRR, NDCG@k (before and after reranking),
    and per-row details for breakdown analysis.

    Note:
    For fair reranker evaluation, keep the same FAISS candidate pool for both
    FAISS and reranked metrics. Optional heuristics (claims filtering) can be
    enabled explicitly via apply_claims_filter.
    """
    results_per_row = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
        question = row['question']
        gold_chunk_id = str(row['chunk_id'])

        # Retrieve top-k candidates from FAISS
        retrieved = retrieve_chunks(
            question, embed_model, faiss_index,
            all_chunks, chunk_metadata, top_k=top_k_retrieve
        )
        retrieved_ids = [r['chunk_id'] for r in retrieved]

        # Keep reranker candidates aligned with FAISS metrics by default.
        candidates_for_rerank = retrieved
        if apply_claims_filter:
            candidates_for_rerank = filter_claims_processing(retrieved, question)

        # Rerank to top-5 (no dedup for fair eval comparison)
        reranked = rerank_results(
            question,
            candidates_for_rerank,
            reranker_model,
            top_n=top_n_rerank,
            deduplicate=False
        )
        reranked_ids = [r['chunk_id'] for r in reranked]

        # --- Recall@k (before reranking) ---
        hit_at_5  = 1 if gold_chunk_id in retrieved_ids[:5]  else 0
        hit_at_10 = 1 if gold_chunk_id in retrieved_ids[:10] else 0
        hit_at_20 = 1 if gold_chunk_id in retrieved_ids[:20] else 0

        # --- MRR (before reranking) ---
        if gold_chunk_id in retrieved_ids:
            mrr = 1.0 / (retrieved_ids.index(gold_chunk_id) + 1)
        else:
            mrr = 0.0

        # --- NDCG@k (before reranking) ---
        ndcg_5  = ndcg_at_k(gold_chunk_id, retrieved_ids, 5)
        ndcg_10 = ndcg_at_k(gold_chunk_id, retrieved_ids, 10)
        ndcg_20 = ndcg_at_k(gold_chunk_id, retrieved_ids, 20)

        # --- Recall@5 after reranking ---
        hit_at_5_reranked = 1 if gold_chunk_id in reranked_ids else 0

        # --- MRR after reranking ---
        if gold_chunk_id in reranked_ids:
            mrr_reranked = 1.0 / (reranked_ids.index(gold_chunk_id) + 1)
        else:
            mrr_reranked = 0.0

        # --- NDCG@5 after reranking ---
        ndcg_5_reranked = ndcg_at_k(gold_chunk_id, reranked_ids, 5)

        results_per_row.append({
            'qa_id':              row.get('qa_id', ''),
            'question_type':      row.get('question_type', ''),
            'doc_type':           row.get('doc_type', ''),
            'coverage_status':    row.get('coverage_status', ''),
            'hit_at_5':           hit_at_5,
            'hit_at_10':          hit_at_10,
            'hit_at_20':          hit_at_20,
            'mrr':                mrr,
            'ndcg_5':             ndcg_5,
            'ndcg_10':            ndcg_10,
            'ndcg_20':            ndcg_20,
            'hit_at_5_reranked':  hit_at_5_reranked,
            'mrr_reranked':       mrr_reranked,
            'ndcg_5_reranked':    ndcg_5_reranked,
        })

    results_df = pd.DataFrame(results_per_row)

    summary = {
        'Recall@5 (FAISS)':    results_df['hit_at_5'].mean(),
        'Recall@10 (FAISS)':   results_df['hit_at_10'].mean(),
        'Recall@20 (FAISS)':   results_df['hit_at_20'].mean(),
        'MRR (FAISS)':         results_df['mrr'].mean(),
        'NDCG@5 (FAISS)':      results_df['ndcg_5'].mean(),
        'NDCG@10 (FAISS)':     results_df['ndcg_10'].mean(),
        'NDCG@20 (FAISS)':     results_df['ndcg_20'].mean(),
        'Recall@5 (reranked)': results_df['hit_at_5_reranked'].mean(),
        'MRR (reranked)':      results_df['mrr_reranked'].mean(),
        'NDCG@5 (reranked)':   results_df['ndcg_5_reranked'].mean(),
    }

    return summary, results_df


print("Evaluation function defined.")

In [89]:
# Also load the pre-trained reranker for comparison
pretrained_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
print("Pre-trained reranker loaded for comparison.")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Pre-trained reranker loaded for comparison.


In [95]:
print("Evaluating PRE-TRAINED bi-encoder + PRE-TRAINED reranker...\n")
pt_summary, pt_details = evaluate_retrieval(
    pretrained_embed_model, pretrained_index, pretrained_reranker,
    all_chunks, chunk_metadata, eval_df
)

print("\n--- Pre-trained Results ---")
for metric, value in pt_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

Evaluating PRE-TRAINED bi-encoder + PRE-TRAINED reranker...



Evaluating: 100%|██████████| 500/500 [01:02<00:00,  7.97it/s]


--- Pre-trained Results ---
  Recall@5 (FAISS)         : 0.6460
  Recall@10 (FAISS)        : 0.7240
  Recall@20 (FAISS)        : 0.8000
  MRR (FAISS)              : 0.4969
  Recall@5 (reranked)      : 0.6460
  MRR (reranked)           : 0.4630


In [96]:
print("Evaluating FINE-TUNED bi-encoder + FINE-TUNED reranker...\n")
ft_summary, ft_details = evaluate_retrieval(
    finetuned_embed_model, finetuned_index, finetuned_reranker,
    all_chunks, chunk_metadata, eval_df
)

print("\n--- Fine-tuned Results ---")
for metric, value in ft_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

Evaluating FINE-TUNED bi-encoder + FINE-TUNED reranker...



Evaluating: 100%|██████████| 500/500 [01:01<00:00,  8.12it/s]


--- Fine-tuned Results ---
  Recall@5 (FAISS)         : 0.7220
  Recall@10 (FAISS)        : 0.7620
  Recall@20 (FAISS)        : 0.8140
  MRR (FAISS)              : 0.5323
  Recall@5 (reranked)      : 0.6740
  MRR (reranked)           : 0.4774


### Step 5.3 — Comparison table

In [97]:
print("\n" + "=" * 75)
print("RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned")
print("=" * 75)
print(f"{'Metric':30s} | {'Pre-trained':>12s} | {'Fine-tuned':>12s} | {'Delta':>10s}")
print("-" * 75)

for metric in pt_summary:
    pt_val = pt_summary[metric]
    ft_val = ft_summary[metric]
    delta = ft_val - pt_val
    sign = '+' if delta >= 0 else ''
    print(f"  {metric:28s} | {pt_val:>11.4f}  | {ft_val:>11.4f}  | {sign}{delta:>9.4f}")

print("=" * 75)


RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned
Metric                         |  Pre-trained |   Fine-tuned |      Delta
---------------------------------------------------------------------------
  Recall@5 (FAISS)             |      0.6460  |      0.7220  | +   0.0760
  Recall@10 (FAISS)            |      0.7240  |      0.7620  | +   0.0380
  Recall@20 (FAISS)            |      0.8000  |      0.8140  | +   0.0140
  MRR (FAISS)                  |      0.4969  |      0.5323  | +   0.0353
  Recall@5 (reranked)          |      0.6460  |      0.6740  | +   0.0280
  MRR (reranked)               |      0.4630  |      0.4774  | +   0.0144


### Step 5.4 — Breakdown by document type and question type

In [ ]:
def print_breakdown(details_df, group_col, metric_col, label):
    """Print a breakdown of a metric by a grouping column."""
    print(f"\n--- {label} ---")
    grouped = details_df.groupby(group_col)[metric_col].agg(['mean', 'count'])
    grouped.columns = [metric_col, 'Count']
    grouped = grouped.sort_values('Count', ascending=False)
    print(grouped.to_string())

print("\n" + "=" * 60)
print("FINE-TUNED MODEL BREAKDOWN")
print("=" * 60)

print_breakdown(ft_details, 'doc_type', 'hit_at_5_reranked', 'Recall@5 (reranked) by Document Type')
print_breakdown(ft_details, 'question_type', 'hit_at_5_reranked', 'Recall@5 (reranked) by Question Type')
print_breakdown(ft_details, 'coverage_status', 'hit_at_5_reranked', 'Recall@5 (reranked) by Coverage Status')

# NDCG breakdowns
print_breakdown(ft_details, 'doc_type', 'ndcg_5_reranked', 'NDCG@5 (reranked) by Document Type')
print_breakdown(ft_details, 'question_type', 'ndcg_5_reranked', 'NDCG@5 (reranked) by Question Type')
print_breakdown(ft_details, 'coverage_status', 'ndcg_5_reranked', 'NDCG@5 (reranked) by Coverage Status')
